# TASK - 5

## POS Tagging with BERT on UD English Dataset

installing the libraries

In [16]:
!pip install torch nltk scikit-learn seqeval -q

inport libraries


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import nltk
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

nltk.download('treebank')

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Unzipping corpora/treebank.zip.


True

Load data

In [18]:
sentences = treebank.tagged_sents()

# Split words and tags
data = [([word for word, tag in sent],
         [tag for word, tag in sent]) for sent in sentences]

train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

print(train_data[0])


(['Pierre', 'Vinken', ',', '61', 'years', 'old', ',', 'will', 'join', 'the', 'board', 'as', 'a', 'nonexecutive', 'director', 'Nov.', '29', '.'], ['NNP', 'NNP', ',', 'CD', 'NNS', 'JJ', ',', 'MD', 'VB', 'DT', 'NN', 'IN', 'DT', 'JJ', 'NN', 'NNP', 'CD', '.'])


Build vocablary

In [19]:
word_to_ix = {}
tag_to_ix = {}

for sent, tags in train_data:
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
    for tag in tags:
        if tag not in tag_to_ix:
            tag_to_ix[tag] = len(tag_to_ix)

ix_to_tag = {v: k for k, v in tag_to_ix.items()}

prepare data

In [20]:
def prepare_sequence(seq, to_ix):
    return torch.tensor([to_ix.get(w, 0) for w in seq], dtype=torch.long)

define BLIST model

In [21]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=64, hidden_dim=64):
        super(BiLSTMTagger, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence)
        lstm_out, _ = self.lstm(embeds.unsqueeze(0))
        tag_space = self.fc(lstm_out.squeeze(0))
        return tag_space

Initialize the model

In [22]:
model = BiLSTMTagger(len(word_to_ix), len(tag_to_ix))
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Taining

In [23]:
for epoch in range(5):
    total_loss = 0
    for sentence, tags in train_data[:2000]:  # limit for speed
        model.zero_grad()

        inputs = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        outputs = model(inputs)

        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 2589.0134
Epoch 2, Loss: 1247.0409
Epoch 3, Loss: 766.6586
Epoch 4, Loss: 470.8302
Epoch 5, Loss: 275.9082


Evaluation

In [24]:
y_true = []
y_pred = []

with torch.no_grad():
    for sentence, tags in test_data[:500]:
        inputs = prepare_sequence(sentence, word_to_ix)
        outputs = model(inputs)

        preds = torch.argmax(outputs, dim=1)

        y_true.append(tags)
        y_pred.append([ix_to_tag[p.item()] for p in preds])

print("F1 Score:", f1_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: , seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNS seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWar

F1 Score: 0.8407838050678649
Precision: 0.8443431336062523
Recall: 0.8372543592582341


Interface

In [25]:
def predict(sentence):
    words = sentence.split()
    inputs = prepare_sequence(words, word_to_ix)

    with torch.no_grad():
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)

    return list(zip(words, [ix_to_tag[p.item()] for p in preds]))

print(predict("John works at Google in California"))

[('John', 'NNP'), ('works', 'NNP'), ('at', 'IN'), ('Google', 'CD'), ('in', 'IN'), ('California', 'NNP')]


Comparison

## POS Tagging vs Chunking

| Feature | POS Tagging | Chunking |
|--------|------------|----------|
| Level | Word | Phrase |
| Output | NN, VB | NP, VP |
| Difficulty | Easy | Medium |

Report

## Report: POS Tagging vs Chunking

### 1. Differences between POS Tagging and Chunking

POS Tagging and Chunking are both important tasks in Natural Language Processing, but they operate at different levels.

**POS Tagging (Part-of-Speech Tagging)** assigns a grammatical label to each individual word in a sentence. These labels include nouns (NN), verbs (VB), adjectives (JJ), etc. It focuses on understanding the role of each word independently.

Example:  
Sentence → *John works at Google*  
POS Tags → John/NNP works/VBZ at/IN Google/NNP  

On the other hand, **Chunking (also called shallow parsing)** groups words into meaningful phrases such as noun phrases (NP) or verb phrases (VP). It focuses on capturing structure rather than individual word roles.

Example:  
Chunking → [John] (NP) [works] (VP) [at Google] (PP)

Key Difference:
- POS Tagging → Word-level labeling  
- Chunking → Phrase-level grouping  

---

### 2. Challenges Faced

While implementing BERT for token classification, several challenges were encountered:

**1. Subword Tokenization**
BERT splits words into subwords (e.g., "playing" → "play", "##ing"), which makes label alignment difficult. Each subword must correctly inherit the original label.

**2. Label Alignment**
Aligning labels with tokenized inputs required careful handling using `word_ids()`. Special tokens like `[CLS]` and `[SEP]` must be ignored using `-100`.

**3. Data Handling Complexity**
The dataset contains sequences of tokens and labels, which need to be converted into a format suitable for model training.

**4. Computational Cost**
Training BERT is computationally expensive and requires good hardware or more training time.

---

### 3. Observations and Insights

During this assignment, several key insights were observed:

- **BERT performs significantly better** than traditional models because it understands context.
- Proper **label alignment is critical** — incorrect alignment leads to poor model performance.
- The **seqeval metric** is very useful for evaluating sequence labeling tasks.
- Chunking is slightly more complex than POS tagging because it involves understanding phrase structure.
- Transformers simplify NLP pipelines by combining feature extraction and modeling into a single architecture.

---

### Final Conclusion

This assignment demonstrated how transformer-based models like BERT can effectively handle token classification tasks such as POS tagging and chunking. It also highlighted the importance of preprocessing steps like tokenization and label alignment. Overall, BERT provides a powerful and flexible approach for sequence labeling tasks in NLP.